# ReMDM: Discrete Diffusion Planning in Craftax

> **Self-contained Colab notebook.** Loads pre-trained checkpoints from a public
> HuggingFace repo and demonstrates live inference, agent visualisation and the
> ReMDM denoising loop. No training is performed inside the notebook.

This notebook accompanies the paper *Return-Weighted ELBO Fine-Tuning Degrades
Masked Diffusion Planners*. It demonstrates a Remasking Discrete Diffusion Model
(**ReMDM**) acting as a planner in Craftax, and reproduces the empirical core of
the paper's Craftax Classic results: **fine-tuning the imitation-pretrained
planner with a return-weighted ELBO makes it worse, no condition recovers the
checkpoint it started from, and the damage is not attributable to the return
weighting.**

**How to use this notebook**

1. Open in Google Colab (preferably with a GPU runtime).
2. Run all cells top-to-bottom.
3. To test on **unseen inputs**, edit the constants in the *Configuration*
   cell (`SEED`, `ENV_NAME`, `EVAL_STEPS`, `EVAL_NUM_ENVS`) and re-run from
   there. Every Craftax seed produces a procedurally generated world the agent
   has never seen.

**Which evaluation path this notebook uses**

Cell 5 evaluates by importing `build_eval_fn` from the ablation harness — the
same function that produced every Craftax number in the paper. It samples with
`sample_plan` (no locked prefix), executes `EVAL_REPLAN` actions per plan at
`VAL_DIFFUSION_STEPS` denoising steps, and scores `returned_episode_returns`.

The alternative path, `main.py --mode inference`, samples with
`sample_plan_inpainting`, which freezes every executed action as an inpainting
prefix. It is a different planner at evaluation time and scores far lower on the
same weights. It is preserved as an explicit ablation in **Cell 5b** and is not
the source of any paper number.

**Scope**

- Cell 3 downloads everything from a single public HuggingFace repo
  (`HF_REPO_ID`) — no authentication required.
- The pre-trained checkpoint is loaded; no training happens here.
- Live evaluation (Cells 5–7) reproduces the paper's evaluation path.
- Pre-computed ablation figures and tables (Cells 10–11) carry the research
  finding.

In [ ]:
# =============================================================================
# CONFIGURATION — edit any of these and re-run from here
# =============================================================================

# Public HuggingFace repo holding the released artefacts: source, checkpoints
# and the pre-computed ablation outputs. No authentication required. Point it
# at your own Hub repo holding the same layout, or train from source
# (see README.md).
HF_REPO_ID = "MathisW78/remdm-craftax"
LOCAL_DIR = "remdm-craftax"

# --- Reproducibility ---
SEED = 42

# "Craftax-Classic-Symbolic-v1": 22 achievements, 17 actions  (diffusion + PPO)
# "Craftax-Symbolic-v1":         65 achievements, 43 actions  (PPO expert only)
ENV_NAME = "Craftax-Classic-Symbolic-v1"

# --- Evaluation protocol -----------------------------------------------------
# These mirror experiments/rl_finetuning/configs/ablations_default.yaml, which
# is the protocol that produced every Craftax number in the paper. The paper's
# own run raised the first two (ablations_final_craftax_classic_gpu_24gb.yaml:
# 192 envs, 1024 steps); the defaults below are the same protocol at a size
# that fits a single Colab GPU. Lower them for speed, but a changed protocol
# no longer corresponds to a paper number.
EVAL_NUM_ENVS = 64          # parallel envs (paper run: 192)
EVAL_STEPS = 512            # env steps per eval rollout (paper run: 1024)
VAL_DIFFUSION_STEPS = 50    # reverse denoising steps per plan (paper: 50)
EVAL_REPLAN = 8             # env steps executed per plan before replanning

# --- Inpainting ablation (Cell 5b) -------------------------------------------
# Denoising budget for the historical-inpainting sampler, which is an ablation
# and NOT the protocol behind any paper number. See Cell 5b.
INPAINTING_DIFFUSION_STEPS = 10

## 1. Project overview

### Problem

Plan action sequences in **Craftax**, a JAX-accelerated, procedurally generated
open-world survival game (a Crafter reimplementation extended with NetHack-like
mechanics). Two variants are reachable from this notebook:

| Environment | Achievements | Actions | Notes |
|---|---|---|---|
| `Craftax-Classic-Symbolic-v1` | 22 | 17 | Crafter ported to JAX |
| `Craftax-Symbolic-v1`         | 65 | 43 | + NetHack mechanics, 9 floors |

Every result in the paper is on **Craftax Classic**. The full-Craftax path in this
notebook runs the PPO expert only — no diffusion checkpoint exists for it, because
the action vocabularies differ (17 vs 43).

### Approach — ReMDM as a non-myopic planner

A bidirectional **DenoisingTransformer** generates a `plan_horizon = 32` action
plan by iteratively denoising masked discrete tokens (MDLM / **ReMDM** —
Wang et al. 2025) conditioned on the current symbolic observation. The agent
executes a short prefix of each plan and replans from the new observation; under
the paper's evaluation protocol that prefix is **8 actions**, and each replan is
a fresh plan conditioned only on the current observation.

**Architecture (Craftax Classic).** Craftax observations are dense and
fog-obscured, so an early-fusion MLP compresses the flattened observation into a
single context token `o_tok ∈ R^384`. That token and a sinusoidal timestep
embedding prefix the action sequence into a 6-layer bidirectional transformer
(`d_model=384`, `n_heads=8`, `d_ff=768`, 2-layer observation MLP encoder of width
768, **cosine** noise schedule, dropout 0.1). The **rescale** remasking strategy is
used at sampling time with η=0.5 and temperature=0.5.

### Pipeline (offline, before this notebook)

```
[1] PPO-RNN expert                (Craftax_Baselines/ppo_rnn.py)
[2] Offline Behavioural Cloning   (main.py --mode offline)
[3] Online DAgger fine-tuning     (main.py --mode online)
[4] Return-weighted ELBO fine-tuning — 25 ablation conditions, 3 seeds
```

The diffusion planner is trained first by **Offline BC** on PPO rollouts, then
separately by **Online DAgger** with exponentially-decaying expert mixing.
Fine-tuning always starts from the converged DAgger checkpoint, which is the
checkpoint this notebook ships and evaluates.

### The objective under study

Fine-tuning runs on the planner's **own rollouts**. Each iteration collects
trajectories under the current policy and cuts them into `H`-step windows. Window
`i` is assigned the return `R_i` — the reward summed over exactly the `H` steps that
window trains on, *not* the episode total broadcast to every window — and receives a
scalar weight

```
A_i = clip( max(R_i, 0) / (µ_batch + ε),  c_min, c_max ),   [c_min, c_max] = [0.1, 5.0]
```

The training loss is the ELBO of the denoising objective with each window's term
scaled by `A_i`, treated as a constant under a stop-gradient. This is a **clipped
return ratio**, in the lineage of reward-weighted regression — it is *not* the
advantage-weighted regression weight, which subtracts a baseline and applies an
exponential temperature `exp(A/β)`. The paper's conclusions characterise this weight
function and do not carry over automatically to the exponential form, which was not
tested.

### The exact decomposition

Write `Ā` for the mean weight and `δ_i = A_i/Ā − 1`, so that `Σ_i δ_i = 0`. The
gradient then factors **exactly**, with no approximation:

```
∇L_RW  =  Ā · [  ∇L_BC   +   (1/B) Σ_i δ_i ∇ℓ_i  ]
                 ‾‾‾‾‾‾        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
                imitation      g_δ — the return's entire contribution
```

`Ā` is a scalar that rescales the step size and leaves the direction unchanged, so
everything the return does enters through `g_δ`, whose RMS scale is the weights'
coefficient of variation `CV_A = sqrt(B/ESS − 1)`. Both terms are measurable at a
single parameter point, which is what makes the central control below possible.

### Headline finding (Craftax Classic)

**Fine-tuning degrades the checkpoint.** From a pretrained score of **11.81**,
baseline RL falls to **8.22 ± 0.14** in 500 iterations — giving up **3.59 points** —
and **none of the 25 conditions** ends above the checkpoint it started from. The
best is `lora` at **11.63 ± 0.03**, which is 0.18 short of never having trained.
The worst is `normalized_adv` at **3.73**.

**This is not reward hacking under an evaluation mismatch.** Collection happens at
5 denoising steps and evaluation at 50 with a different replanning interval, so in
principle eval could fall while the objective quietly succeeded. It does not: mean
episodic return of the *collected* rollouts falls from **5.21 to 3.73** over the
same 500 iterations in which eval score falls from **12.06 to 8.49**. Both fall
together.

**The loss is concentrated in the deep tech tree.** Mean completion rate by tier
goes from 63%, 67%, 69%, 23% and 0.1% for the checkpoint to 52%, 51%, 47%, 10% and
0% after fine-tuning. Individually, `collect_stone` falls 92% → 68%,
`make_stone_pickaxe` 70% → 45%, `collect_iron` 41% → 21% and `make_iron_sword`
23% → 8%, while shallow behaviours hold (`collect_wood` 98%, `collect_sapling`
rises slightly). Fine-tuning removes capability the checkpoint had.

**And the reward is not what does it.** The return term is large and points away
from imitation — `‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine `0.02 ± 0.05`, against
`0.893 ± 0.010` between two independent noise draws of `∇L_BC` itself. But
suppressing it does not help: advantage clipping cuts the return term fivefold, to
`0.097 ± 0.003`, and scores **3.16 points below** the unclipped baseline. A
condition with almost no return term degrades *further* than one with a large one.
What orders the suite is how much plasticity each condition allows.

This is mirrored by the MiniHack PyTorch codebase, where the same suite takes the
checkpoint from **47.5% to 43.8%** (seed std 6.1 points). That effect is small and
three conditions finish nominally above the checkpoint, so the paper reports it as
a matching ordering rather than as a second confirmation.

The notebook demonstrates:

| Cell | Demonstrates |
|---|---|
| 5  | Live evaluation through the paper's own evaluation path |
| 5b | The historical-inpainting sampler, as a labelled ablation |
| 6  | The agent meaningfully interacts with Craftax (rewards, achievements) |
| 7  | The ReMDM iterative-unmasking sampler in action |
| 8  | The PPO-RNN expert the planner was distilled from |
| 10–11 | Pre-computed ablation results — what fine-tuning does, and what orders it |

In [ ]:
# =============================================================================
# Install pinned dependencies, download the HF repo, set up sys.path
# =============================================================================

import importlib
import os
import subprocess
import sys


def _pip(*pkgs: str) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        stdout=subprocess.DEVNULL,
    )


# On Colab, installing flax/optax/craftax can upgrade JAX as a transitive dep
# but leave the CUDA PJRT plugin at the old version (-> PJRT size mismatch).
# Fix: include jax[cuda12] in the install so pip keeps the plugin in sync.
_on_colab = "google.colab" in sys.modules or os.path.exists("/content")

_pip(
    "jax[cuda12]" if _on_colab else "jax>=0.9.2",
    "huggingface_hub>=1.9.1",
    "craftax>=1.5.0",
    "flax>=0.12.6",
    "optax>=0.2.8",
    "orbax-checkpoint>=0.12",
    "distrax>=0.1.7",
    "chex>=0.1.91",
    "polars>=1.39.3",
    "orjson>=3.11.8",
    "pyyaml>=6.0",
)
importlib.invalidate_caches()

import jax  # noqa: F401
import craftax  # noqa: F401

backend = jax.default_backend()
device = jax.devices()[0]
print(f"JAX {jax.__version__} | backend={backend} | device={device}")
if backend != "gpu":
    print(
        "WARNING: JAX is running on CPU. Inference will be ~10x slower than GPU."
        "\n         In Colab, switch to a GPU runtime via Runtime -> Change runtime type."
    )

# ----------------------------------------------------------------------------
# Pull the project tree (code, configs, checkpoints, pre-computed results)
# from a single public HuggingFace repo. No authentication required.
# ----------------------------------------------------------------------------
from huggingface_hub import snapshot_download

if HF_REPO_ID == "UNSET_HF_REPO_ID":
    raise RuntimeError(
        "HF_REPO_ID is unset. Edit the constant in Cell 1 to point at a Hub "
        "repo holding the published layout, or train from source "
        "(see README.md)."
    )

snapshot_path = snapshot_download(repo_id=HF_REPO_ID, local_dir=LOCAL_DIR)
print(f"Snapshot downloaded to: {snapshot_path}")

# Both the parent (for `import src.planners...`) and the Craftax_Baselines/
# subdir (for `from wrappers import ...` inside Craftax_Baselines/ppo_rnn.py)
# must be on sys.path. main.py does the same thing.
for p in (snapshot_path, os.path.join(snapshot_path, "Craftax_Baselines")):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(snapshot_path)
print(f"cwd: {os.getcwd()}")


In [ ]:
# =============================================================================
# Load the pre-trained DAgger diffusion checkpoint
# =============================================================================

import json

import jax
import jax.numpy as jnp
from craftax.craftax_env import make_craftax_env_from_name

from src.planners.model import build_model, load_checkpoint, make_apply_fns

# Checkpoint inventory bundled with the HF repo.
DIFFUSION_OFFLINE_CKPT = (
    "checkpoints/offline/Craftax-Classic-Symbolic-v1-Offline-Diffusion-BC-100M"
)
DIFFUSION_ONLINE_CKPT = (
    "checkpoints/online/Craftax-Classic-Symbolic-v1-Online-Diffusion-DAgger-100M"
)
PPO_CKPT = {
    "Craftax-Classic-Symbolic-v1": (
        "checkpoints/ppo_agents/Craftax-Classic-Symbolic-v1-PPO_RNN-1000M"
    ),
    "Craftax-Symbolic-v1": (
        "checkpoints/ppo_agents/Craftax-Symbolic-v1-PPO_RNN-1000M"
    ),
}

# We pin to the DAgger (online) checkpoint — that is the headline result.
# The architecture hyperparameters are stored alongside the offline checkpoint
# in `resume_metadata.json` (online and offline share the same architecture).
with open(os.path.join(DIFFUSION_OFFLINE_CKPT, "resume_metadata.json")) as f:
    META = json.load(f)
ARCH_CFG = META["config_snapshot"]

# The diffusion checkpoints are trained on Classic Craftax. The denoiser's
# action head dimension is fixed at training time (num_actions = 17 for Classic),
# so we cannot transfer it to Full Craftax (43 actions). PPO covers both.
DIFFUSION_ENV_NAME = "Craftax-Classic-Symbolic-v1"

env_init = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
env_params_init = env_init.default_params
NUM_ACTIONS = int(env_init.action_space(env_params_init).n)
OBS_DIM = int(env_init.observation_space(env_params_init).shape[0])
PLAN_HORIZON = int(ARCH_CFG["PLAN_HORIZON"])

print(f"DiffusionEnv : {DIFFUSION_ENV_NAME}")
print(f"  obs_dim    : {OBS_DIM}")
print(f"  num_actions: {NUM_ACTIONS}")
print(f"Architecture : d_model={ARCH_CFG['D_MODEL']} n_layers={ARCH_CFG['N_LAYERS']} "
      f"n_heads={ARCH_CFG['N_HEADS']} plan_horizon={PLAN_HORIZON}")

model = build_model(ARCH_CFG, NUM_ACTIONS)
apply_eval, _ = make_apply_fns(model)
diffusion_params = load_checkpoint(
    model,
    jax.random.PRNGKey(SEED),
    OBS_DIM,
    PLAN_HORIZON,
    DIFFUSION_ONLINE_CKPT,
)

n_params = sum(int(p.size) for p in jax.tree.leaves(diffusion_params))
print(f"Loaded DAgger checkpoint: {n_params / 1e6:.2f}M parameters")


In [ ]:
# =============================================================================
# CELL 5 (PRIORITY) — live evaluation through the paper's evaluation path
# =============================================================================
#
# This calls `experiments/rl_finetuning/ablations/training.py::build_eval_fn`
# directly — the *same* function that produced every Craftax Classic number in
# the paper. It is imported rather than re-implemented so the notebook and the
# ablation harness cannot drift apart.
#
# What that path does, and why it matters:
#   * `sample_plan(...)` is called with no locked prefix, so every replan is a
#     fresh plan conditioned only on the current observation — genuine MPC.
#   * `EVAL_REPLAN` (8) actions are executed per plan, at
#     `VAL_DIFFUSION_STEPS` (50) denoising steps.
#   * The score is `returned_episode_returns`: the mean return over episodes
#     that actually *terminated* inside the rollout, weighted by
#     `returned_episode`. It is not a first-life-only statistic.
#
# `main.py --mode inference` takes a different path (`sample_plan_inpainting`,
# which locks every executed action as an inpainting prefix). That path scores
# far lower on the same checkpoint and is NOT the source of any paper number;
# Cell 5b runs it deliberately, as an ablation, so the two can be compared.
#
# Expected value: the pretrained (DAgger) checkpoint scores 11.81 in the paper
# at 192 envs / 1024 steps. At the smaller default protocol below expect the
# same regime with more seed noise.
# -----------------------------------------------------------------------------

import time

from experiments.rl_finetuning.ablations.training import build_eval_fn
from src.planners.env import make_env

if "Classic" not in ENV_NAME:
    print(
        f"ENV_NAME={ENV_NAME!r}: no diffusion checkpoint exists for Full Craftax\n"
        f"(action vocabularies differ: 17 vs 43). Skipping diffusion eval — see\n"
        f"Cell 8 for the matching PPO expert evaluation on this environment."
    )
else:
    # Config keys build_eval_fn reads. Architecture and sampling knobs come
    # from the checkpoint's own snapshot; the protocol knobs come from Cell 1.
    eval_cfg = {
        **{k: v for k, v in ARCH_CFG.items() if k.isupper()},
        "ENV_NAME": DIFFUSION_ENV_NAME,
        "NUM_ACTIONS": NUM_ACTIONS,
        "PLAN_HORIZON": PLAN_HORIZON,
        "EVAL_STEPS": EVAL_STEPS,
        "EVAL_REPLAN": EVAL_REPLAN,
        "VAL_DIFFUSION_STEPS": VAL_DIFFUSION_STEPS,
    }

    # Same wrapped env the harness evaluates in: LogWrapper supplies the
    # `returned_episode` / `returned_episode_returns` fields the score uses.
    eval_env, eval_env_params = make_env(eval_cfg, EVAL_NUM_ENVS)

    eval_policy = build_eval_fn(eval_env, eval_env_params, apply_eval, eval_cfg)

    n_cycles = EVAL_STEPS // EVAL_REPLAN
    print(
        f"Evaluating {EVAL_NUM_ENVS} envs x {EVAL_STEPS} steps "
        f"({n_cycles} plan-execute cycles, {EVAL_REPLAN} actions per plan, "
        f"{VAL_DIFFUSION_STEPS} denoising steps per plan)...\n"
        f"First call includes JIT compilation."
    )

    t0 = time.time()
    metrics = eval_policy(diffusion_params, jax.random.PRNGKey(SEED))
    metrics = jax.tree.map(lambda x: float(x), metrics)
    elapsed = time.time() - t0

    score = metrics["returned_episode_returns"]
    print(f"\n{'=' * 62}")
    print(f"EVALUATION COMPLETE ({elapsed:.1f}s)")
    print(f"{'=' * 62}")
    print(f"Score (returned_episode_returns) : {score:.2f}")
    print(f"Mean episode length              : {metrics['returned_episode_lengths']:.1f}")
    print(f"\nPaper reference (192 envs, 1024 steps): pretrained checkpoint = 11.81")

    # Achievement rates on the same rollout, where the harness logs them.
    ach = {
        k.split("/")[-1]: v
        for k, v in metrics.items()
        if k.startswith("Achievements/")
    }
    if ach:
        majority = sum(1 for v in ach.values() if v >= 0.5)
        print(f"\nAchievements completed in a majority of episodes: {majority} of {len(ach)}")
        print("(paper's pre-computed table for this checkpoint: 13 of 22)\n")
        for name, rate in sorted(ach.items(), key=lambda kv: -kv[1]):
            print(f"  {name:24s} {rate * 100:5.1f}%")

In [ ]:
# =============================================================================
# CELL 5b — ABLATION: the historical-inpainting sampler (not a paper protocol)
# =============================================================================
#
# `main.py --mode inference` evaluates through `sample_plan_inpainting`, which
# locks every already-executed action as a fixed inpainting prefix (Diffuser
# Sec. 3.3). It replans every single step, but because positions
# `0..hist_len-1` are frozen, at step k of a 32-action window only `32 - k`
# positions are still free. As the window fills, the sampler has progressively
# less left to decide, so execution tends towards open-loop rather than the
# genuine MPC that Cell 5 runs.
#
# This is kept, and kept runnable, as an ablation on the planning-as-inpainting
# design choice. It is NOT how any number in the paper was produced. Comparing
# its output with Cell 5 on the same checkpoint is the point of the cell: the
# gap between them is attributable to the sampler and the scoring rule, not to
# the weights.
#
# Off by default because it is slow (it replans on every env step).
# -----------------------------------------------------------------------------

RUN_INPAINTING_ABLATION = False  # set True to run the comparison

if not RUN_INPAINTING_ABLATION:
    print(
        "Inpainting ablation skipped. Set RUN_INPAINTING_ABLATION = True to run it.\n"
        "For reference, on the released DAgger checkpoint the two paths give:\n"
        "  harness path (Cell 5, sample_plan, 50 steps, replan every 8) : 11.81\n"
        "  inpainting path (this cell, 10 steps, replan every step)     :  3.26\n"
        "Same weights; the difference is the evaluation path."
    )
elif "Classic" not in ENV_NAME:
    print(f"Skipping inpainting ablation: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    from src.planners.inference import run_inference

    run_inference(
        {
            **{k: v for k, v in ARCH_CFG.items() if k.isupper()},
            "ENV_NAME": DIFFUSION_ENV_NAME,
            "SEED": SEED,
            "EVAL_STEPS": EVAL_STEPS,
            "EVAL_NUM_ENVS": EVAL_NUM_ENVS,
            "DIFFUSION_STEPS_EVAL": INPAINTING_DIFFUSION_STEPS,
            "INFERENCE_SAMPLER": "inpainting",
            "USE_WANDB": False,
            "CHECKPOINT_PATH": DIFFUSION_ONLINE_CKPT,
        }
    )

In [ ]:
# =============================================================================
# CELL 6 (PRIORITY) — visualise the ReMDM planner acting in Craftax
# =============================================================================
#
# Rolls the planner out on a small batch of envs and captures per-step rewards,
# achievements and actions for plotting.
#
# The control loop here is a faithful copy of the plan-execute cycle inside
# `build_eval_fn` (experiments/rl_finetuning/ablations/training.py): a fresh
# `sample_plan` call with no locked prefix, then `EVAL_REPLAN` actions executed
# from that plan, repeated. It is inlined only because we need the per-step
# trajectories that `build_eval_fn` reduces away; the sampling call and the
# replan cadence are identical, so the two cannot drift.
# -----------------------------------------------------------------------------

import numpy as np

from src.diffusion.schedules import SCHEDULE_MAP

VIZ_SCHEDULE_FN, _ = SCHEDULE_MAP[ARCH_CFG.get("DIFFUSION_SCHEDULE", "cosine")]
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from src.diffusion.sampling import sample_plan

VIZ_NUM_ENVS = 4
VIZ_STEPS = 600

if "Classic" not in ENV_NAME:
    print(f"Skipping behaviour viz: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    viz_env = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
    viz_params = viz_env.default_params

    rng = jax.random.PRNGKey(SEED + 1)
    rng, env_rng = jax.random.split(rng)
    obs0, state0 = jax.vmap(viz_env.reset, in_axes=(0, None))(
        jax.random.split(env_rng, VIZ_NUM_ENVS), viz_params,
    )

    @jax.jit
    def viz_cycle(carry, _):
        """One plan-execute cycle: mirrors build_eval_fn's `_cycle`."""
        obs, state, rng = carry
        rng, plan_rng = jax.random.split(rng)

        # No history / hist_len: the plan is generated fresh from the current
        # observation, exactly as build_eval_fn does.
        plan = sample_plan(
            apply_eval,
            diffusion_params,
            plan_rng,
            obs,
            NUM_ACTIONS,
            PLAN_HORIZON,
            num_steps=VAL_DIFFUSION_STEPS,
            schedule_fn=VIZ_SCHEDULE_FN,
            remask_strategy=ARCH_CFG.get("REMASK_STRATEGY", "rescale"),
            eta=ARCH_CFG.get("ETA", 0.5),
            use_loop=ARCH_CFG.get("USE_LOOP", True),
            t_on=ARCH_CFG.get("T_ON", 0.7),
            t_off=ARCH_CFG.get("T_OFF", 0.3),
            temperature=ARCH_CFG["TEMPERATURE"],
            top_p=ARCH_CFG["TOP_P"],
        )

        def exec_step(inner, step_i):
            obs_i, state_i, r = inner
            r, s_rng = jax.random.split(r)
            action = plan[:, step_i]
            obs_next, state_next, reward, done, _ = jax.vmap(
                viz_env.step, in_axes=(0, 0, 0, None),
            )(jax.random.split(s_rng, VIZ_NUM_ENVS), state_i, action, viz_params)
            return (obs_next, state_next, r), (
                action, reward, done, state_next.achievements,
            )

        (obs, state, rng), out = jax.lax.scan(
            exec_step, (obs, state, rng), jnp.arange(EVAL_REPLAN),
        )
        return (obs, state, rng), out

    n_cycles = VIZ_STEPS // EVAL_REPLAN
    print(
        f"Rolling out {VIZ_NUM_ENVS} agents for {n_cycles * EVAL_REPLAN} steps "
        f"({n_cycles} cycles x {EVAL_REPLAN} actions per plan)..."
    )
    _, (acts, rews, dones, achs) = jax.lax.scan(
        viz_cycle, (obs0, state0, rng), None, n_cycles,
    )
    # Flatten [cycles, replan, envs] -> [T, envs]
    flat = lambda x: np.array(x).reshape(-1, *np.array(x).shape[2:])
    acts_np, rews_np, dones_np, achs_np = flat(acts), flat(rews), flat(dones), flat(achs)
    T = acts_np.shape[0]

    # First-life cumulative reward + unlock count per agent
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cum_reward = np.cumsum(rews_np, axis=0)
    unlock_count = achs_np.sum(axis=-1)
    for e in range(VIZ_NUM_ENVS):
        end = np.where(dones_np[:, e])[0]
        cutoff = int(end[0]) + 1 if len(end) > 0 else T
        axes[0].plot(np.arange(cutoff), cum_reward[:cutoff, e], label=f"agent {e}")
        axes[1].plot(np.arange(cutoff), unlock_count[:cutoff, e], label=f"agent {e}")
    axes[0].set_title("Cumulative reward (first life)")
    axes[0].set_xlabel("env step"); axes[0].set_ylabel("cumulative reward")
    axes[1].set_title("Achievements unlocked")
    axes[1].set_xlabel("env step"); axes[1].set_ylabel("# unique achievements")
    for ax in axes:
        ax.legend(loc="best", fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # Action histogram (which actions does the planner actually use?)
    from craftax.craftax_classic.constants import Action as ClassicAction
    action_names = [a.name for a in ClassicAction]
    counts = np.bincount(acts_np.flatten(), minlength=NUM_ACTIONS)
    order = np.argsort(-counts)
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar(range(NUM_ACTIONS), counts[order])
    ax.set_xticks(range(NUM_ACTIONS))
    ax.set_xticklabels([action_names[i] for i in order], rotation=70, ha="right", fontsize=8)
    ax.set_title(f"Action usage over {T * VIZ_NUM_ENVS} env steps")
    ax.set_ylabel("count"); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# CELL 7 (PRIORITY) — visualise the ReMDM iterative unmasking process
# =============================================================================
#
# `sample_plan` is JIT-compiled via lax.scan, so to capture the intermediate
# token sequences we re-implement its body as a Python loop. This is faithful
# to the sampler Cell 5 evaluates with — only the loop construct differs, and
# `hist_len = 0` means no positions are locked, exactly as build_eval_fn calls it.
# Each row of the heatmap below is one denoising step: at step 0 the entire
# plan_horizon is masked; by the final step the plan has been fully resolved
# into action tokens, with ReMDM-style stochastic remasking interspersed.
# -----------------------------------------------------------------------------

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

if "Classic" not in ENV_NAME:
    print(f"Skipping denoising viz: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    DENOISE_STEPS = 12  # show enough steps to see iterative unmasking
    MASK_ID = NUM_ACTIONS

    # Sample one observation from the env to condition on.
    viz_env = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
    viz_params = viz_env.default_params
    one_rng = jax.random.PRNGKey(SEED + 2)
    obs1, _state1 = viz_env.reset(one_rng, viz_params)
    obs_b = obs1[None, :]                      # [1, obs_dim]

    seq = jnp.full((1, PLAN_HORIZON), MASK_ID, dtype=jnp.int32)
    history = jnp.full((1, PLAN_HORIZON), MASK_ID, dtype=jnp.int32)
    hist_len = jnp.zeros((1,), dtype=jnp.int32)
    rng = jax.random.PRNGKey(SEED + 3)
    temperature = float(ARCH_CFG["TEMPERATURE"])
    top_p = float(ARCH_CFG["TOP_P"])

    trace = [np.array(seq[0])]               # row 0 = fully masked

    # Mirror the body of src.diffusion.sampling.sample_plan._step
    for step in range(1, DENOISE_STEPS + 1):
        rng, model_rng, sample_rng, remask_rng = jax.random.split(rng, 4)
        ratio = step / DENOISE_STEPS
        t_tensor = jnp.full((1,), 1.0 - ratio)
        logits = apply_eval(diffusion_params, obs_b, seq, t_tensor, model_rng) / max(temperature, 1e-8)

        # Nucleus filtering
        probs = jax.nn.softmax(logits, axis=-1)
        sorted_idx = jnp.argsort(-probs, axis=-1)
        sorted_p = jnp.take_along_axis(probs, sorted_idx, axis=-1)
        cutoff = jnp.cumsum(sorted_p, axis=-1) - sorted_p
        inv_idx = jnp.argsort(sorted_idx, axis=-1)
        nucleus_mask = jnp.take_along_axis(cutoff >= top_p, inv_idx, axis=-1)
        logits = jnp.where(nucleus_mask, -jnp.inf, logits)

        preds = jax.random.categorical(sample_rng, logits, axis=-1)
        conf = jnp.take_along_axis(
            jax.nn.softmax(logits, axis=-1), preds[..., None], axis=-1,
        ).squeeze(-1)
        num_unmask = max(1, int(PLAN_HORIZON * ratio))
        sorted_conf = jnp.sort(conf, axis=-1)[..., ::-1]
        thresh = sorted_conf[0, num_unmask - 1]
        seq_new = jnp.where(conf < thresh, MASK_ID, preds)

        # ReMDM-style remasking (matches sample_plan_inpainting)
        remask_prob = 0.15 * (1.0 - ratio)
        do_remask = (
            (jax.random.uniform(remask_rng, seq_new.shape) < remask_prob)
            & (seq_new != MASK_ID)
        )
        seq_new = jnp.where(do_remask, MASK_ID, seq_new)

        # Lock historical prefix (here: empty, so no-op)
        pos = jnp.broadcast_to(jnp.arange(PLAN_HORIZON)[None, :], (1, PLAN_HORIZON))
        seq_new = jnp.where(pos < hist_len[:, None], history, seq_new)

        seq = seq_new
        trace.append(np.array(seq[0]))

    trace = np.stack(trace)                  # [steps+1, plan_horizon]

    # Heatmap: rows = denoising step, columns = action position
    # Mask cells coloured grey, action cells coloured by token id (viridis).
    masked = trace == MASK_ID
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.cm.viridis.copy()
    display = np.where(masked, np.nan, trace.astype(float))
    im = ax.imshow(
        display, aspect="auto", cmap=cmap, vmin=0, vmax=NUM_ACTIONS - 1,
        interpolation="nearest",
    )
    # Overlay grey for masked cells
    ax.imshow(
        np.where(masked, 1.0, np.nan), aspect="auto",
        cmap=ListedColormap(["#dddddd"]), vmin=0, vmax=1, interpolation="nearest",
    )
    ax.set_xlabel("plan position (action token)")
    ax.set_ylabel("denoising step")
    ax.set_title(
        f"ReMDM iterative unmasking ({DENOISE_STEPS} steps, plan_horizon={PLAN_HORIZON})"
        "\nGrey = MASK token, colour = sampled action ID"
    )
    cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("action ID")
    plt.tight_layout(); plt.show()

    n_masked = masked.sum(axis=1)
    print(f"Masked tokens per step: {list(n_masked)}")
    print(f"  step 0  (start): {int(n_masked[0])}/{PLAN_HORIZON} masked")
    print(f"  step {DENOISE_STEPS} (final): {int(n_masked[-1])}/{PLAN_HORIZON} masked")

In [ ]:
# =============================================================================
# CELL 8 — PPO-RNN expert baseline (loaded live, evaluated on `ENV_NAME`)
# =============================================================================
#
# DAgger trains the diffusion planner to imitate this PPO-RNN expert (trained
# in `Craftax_Baselines/`). This cell loads and evaluates the expert itself, so
# the planner in Cell 5 can be read against the policy it was distilled from.
# Note that the two are scored by different harnesses and the paper makes no
# quantitative imitation-gap claim, so treat this as context, not as a
# like-for-like comparison.
# For Full Craftax, where no diffusion checkpoint exists (action vocabularies
# differ: 17 vs 43), this PPO baseline is the only evaluation that can run.
# -----------------------------------------------------------------------------

import os
import numpy as np
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from orbax.checkpoint import checkpoint_utils
from craftax.craftax_env import make_craftax_env_from_name

from src.planners.ppo import build_ppo_network, PPOAgent

PPO_NUM_ENVS = EVAL_NUM_ENVS
PPO_EVAL_STEPS = min(EVAL_STEPS, 1500)  # PPO eval is fast — bound it for snappy demo

ppo_path = os.path.abspath(PPO_CKPT[ENV_NAME])
ppo_env = make_craftax_env_from_name(ENV_NAME, auto_reset=True)
ppo_params_env = ppo_env.default_params
ppo_num_actions = int(ppo_env.action_space(ppo_params_env).n)
ppo_obs_dim = int(ppo_env.observation_space(ppo_params_env).shape[0])

# Build the PPO-RNN network and an abstract param pytree from a dummy init.
PPO_LAYER_SIZE = 512
ppo_net = build_ppo_network("ppo_rnn", ppo_num_actions, PPO_LAYER_SIZE,
                            {"LAYER_SIZE": PPO_LAYER_SIZE})
_dummy_x = (jnp.zeros((1, PPO_NUM_ENVS, ppo_obs_dim)),
            jnp.zeros((1, PPO_NUM_ENVS)))
_abstract_params = ppo_net.init(
    jax.random.PRNGKey(0),
    jnp.zeros((PPO_NUM_ENVS, PPO_LAYER_SIZE)),
    _dummy_x,
)

# Restore params only (the on-disk checkpoint also contains opt_state which we
# don't need for inference). `partial_restore=True` lets us read just `params`,
# and `construct_restore_args` is required on orbax >= 0.11 to provide sharding.
_restore_args = checkpoint_utils.construct_restore_args({"params": _abstract_params})
with ocp.CheckpointManager(ppo_path) as _mgr:
    _step = _mgr.latest_step()
    _restored = _mgr.restore(
        _step,
        args=ocp.args.PyTreeRestore(
            item={"params": _abstract_params},
            restore_args=_restore_args,
            partial_restore=True,
        ),
    )
print(f"Loaded PPO_RNN checkpoint from '{ppo_path}' (step {_step})")

ppo_agent = PPOAgent(
    network=ppo_net,
    params=_restored["params"],
    model_type="ppo_rnn",
    layer_size=PPO_LAYER_SIZE,
)

rng = jax.random.PRNGKey(SEED + 100)
rng, env_rng = jax.random.split(rng)
obs, state = jax.vmap(ppo_env.reset, in_axes=(0, None))(
    jax.random.split(env_rng, PPO_NUM_ENVS), ppo_params_env,
)
hidden0 = ppo_agent.init_hidden(PPO_NUM_ENVS)
done0 = jnp.zeros(PPO_NUM_ENVS, dtype=bool)


@jax.jit
def ppo_step(carry, _):
    obs, state, hidden, done, rng = carry
    rng, act_rng, env_rng = jax.random.split(rng, 3)
    action, hidden = ppo_agent.act(obs, done, hidden, act_rng, temperature=1.0)
    obs_next, state_next, reward, done_next, _ = jax.vmap(
        ppo_env.step, in_axes=(0, 0, 0, None),
    )(jax.random.split(env_rng, PPO_NUM_ENVS), state, action, ppo_params_env)
    return (obs_next, state_next, hidden, done_next, rng), (reward, done_next, state_next.achievements)


print(f"Running PPO expert: {PPO_NUM_ENVS} envs x {PPO_EVAL_STEPS} steps on {ENV_NAME}...")
_, (rewards, dones, achievements) = jax.lax.scan(
    ppo_step, (obs, state, hidden0, done0, rng), jnp.arange(PPO_EVAL_STEPS),
)

rewards_np = np.array(rewards)
dones_np = np.array(dones)
ach_np = np.array(achievements)

# First-life evaluation (matches src/planners/inference.py convention)
ep_returns = np.zeros(PPO_NUM_ENVS)
ep_unlocks = np.zeros(PPO_NUM_ENVS, dtype=int)
for i in range(PPO_NUM_ENVS):
    deaths = np.where(dones_np[:, i])[0]
    end = int(deaths[0]) if len(deaths) > 0 else PPO_EVAL_STEPS - 1
    ep_returns[i] = rewards_np[: end + 1, i].sum()
    ep_unlocks[i] = int(ach_np[: end + 1, i].max(axis=0).sum())

print()
print(f"PPO expert mean return : {ep_returns.mean():.2f}  (best={ep_returns.max():.2f})")
print(f"PPO expert mean unlocks: {ep_unlocks.mean():.2f} achievements")
if "Classic" in ENV_NAME:
    print()
    print(
        "Context for the diffusion planner evaluated in Cell 5.\n"
        "The pre-computed harness table puts the pretrained diffusion\n"
        "checkpoint at 13 of 22 achievements completed in a majority of\n"
        "episodes, falling to 10 of 22 after 500 iterations of baseline RL.\n"
        "The PPO expert above is scored by a different harness on a different\n"
        "rollout budget, so read it as the policy the planner was distilled\n"
        "from rather than as a matched comparison. The paper makes no\n"
        "quantitative imitation-gap claim."
    )

## 2. RL fine-tuning ablation suite (pre-computed)

Once DAgger had plateaued, we ran a **25-condition suite** of **return-weighted ELBO**
fine-tuning interventions on top of it. Every figure and table below was generated
offline by `experiments/rl_finetuning/run_ablations.py` and shipped in the HF repo at
`experiments/rl_finetuning/outputs/craftax_classic_ablations/`.
Each condition runs 500 iterations across 192 parallel envs and 3 seeds, all
initialised from the same checkpoint.

The four groups tested:

| Group | Hypothesis tested |
|---|---|
| **A** Regularisation | KL penalty, EWC, LLRD, LoRA, mixed replay, hard trust region |
| **B** Training signal | low-t, t-curriculum, entropy bonus, PCGrad, advantage clip, std-normalised advantages, BC-on-wins |
| **C** Architectural freezing | frozen backbone, head-only, attention-only, FFN-only, top-k layer ablations |
| **D** Data quality / weight computation | return filtering, action diversity, EMA running stats, learned reward model |

### Group-level result (Craftax Classic, Table 7 from the paper)

| Group | N | Mean | Best | Worst | Std |
|---|---|---|---|---|---|
| Pretrained checkpoint | – | **11.81** | – | – | – |
| Baseline RL           | 1 |  8.22 | – | – | 0.00 |
| A (Regularisation)    | 6 |  9.74 | 11.63 | 8.01 | 1.42 |
| B (Training signal)   | 7 |  7.07 |  8.51 | 3.73 | 1.77 |
| C (Architectural freezing) | 7 | 10.53 | 11.30 | 9.26 | 0.63 |
| D (Data quality / weights) | 4 |  7.97 |  9.75 | 5.56 | 1.51 |

Every group mean is below the pretrained checkpoint.

### Three findings (paper framing)

1. **Every condition degrades, and the suite is ordered by plasticity.** No
   condition recovers 11.81. The top five are exactly the five that most restrict
   the update, in the order of how tightly they restrict it — `lora` 11.63,
   `trust_region_kl` 11.53, `head_only` 11.30, `frozen_backbone` 11.09,
   `layer_ablation_top1` 10.93. That ordering is imposed by the design rather than
   observed after the fact, and the measured version agrees: final KL divergence
   from the checkpoint correlates with final score at Spearman **−0.79** (−0.75
   with LoRA excluded, which matters because the drift probe reads LoRA's frozen
   base weights and records near-zero). The paper is deliberately cautious about
   how much this adds — when every condition degrades, "score falls with distance
   travelled" is close to restating "more training makes it worse faster".
2. **The weights discriminate, and the return term is large.** `CV_A` averages
   **1.16** on Craftax Classic (effective sample size 437 of a 1024-window batch),
   so the returns *do* rank windows within a batch and the familiar explanation
   that sparse returns fail to discriminate is unavailable. Measured directly at
   the pretrained parameters, `‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine
   **0.02 ± 0.05**. A cosine needs a reference: in `D = 9.33 × 10⁶` dimensions two
   independent directions have `cos ≈ 3.3 × 10⁻⁴`, so 0.02 is about sixty standard
   deviations from random — `g_δ` carries structure. But two independent noise
   draws of `∇L_BC` itself agree at **0.893 ± 0.010**, so `g_δ` points somewhere
   imitation does not.
3. **Shrinking the return term makes things worse — the central negative control.**
   If the return term drove the degradation, suppressing it should help. Advantage
   clipping confines the weights to [0.8, 1.2] and does exactly what it should:
   `CV_A` falls from 0.98 to 0.20 and `‖g_δ‖/‖∇L_BC‖` from 0.49 to
   **0.097 ± 0.003**, a fivefold reduction. Its score is **5.06** — 3.16 below
   baseline RL and second worst in the suite. The binary win mask sits between the
   two on both axes (`CV_A` 0.77, ratio 0.417 ± 0.025) and scores 7.28, also below
   baseline. Across these three, the ordering of the return term's magnitude is the
   **reverse** of the ordering of final score. A condition whose return term is
   nearly absent degrades further than one whose return term is large, so the
   degradation cannot be attributed to the return weighting.

### What is left is the data

Advantage clipping is also the nearest thing in the suite to **unweighted training
on the model's own rollouts**. It sits near the bottom. Together with the
plasticity ordering, that points at fine-tuning on self-generated rollouts as the
damaging ingredient, with the return weighting neither causing the harm nor
preventing it. The paper is explicit that this is an inference from a
near-substitute, not a measurement: the unweighted-ELBO-on-all-rollouts arm was
not run.

### The worst condition

`normalized_adv` (**3.73**, −8.08 from the checkpoint) is informative rather than
anomalous. It alone mean-centres the weights, mapping `A_i` to `(A_i − Ā)/std(A)` and
making roughly half of them **negative**, which violates the non-negativity assumption
(A1) the decomposition needs. A negative weight on a cross-entropy term is gradient
*ascent* on that sequence's likelihood — unbounded and without a trust region — and
its drift lands two to four orders of magnitude beyond any other condition. Supplying
the repulsive direction a policy gradient would have does not rescue the objective;
it destroys it. Under mean-centring `Ā ≈ 0`, so `δ_i = A_i/Ā − 1` is undefined and the
measured ratio diverges — a statement about the objective ceasing to be a bound, not
about the size of any return signal.

**Statistics.** With 3 seeds per arm, an exact two-sided permutation test enumerates
C(6,3) = 20 relabellings, so the smallest attainable p-value is 2/20 = **0.10** — a
reported p = 0.10 is the most extreme outcome the design can produce, not a null
result. The shipped test compares the condition furthest from baseline
(`normalized_adv`) and reports p at that floor, with a 95% bootstrap CI on the
difference of **[−5.04, −3.84]**. Cell 11 prints the file verbatim.

In [ ]:
# =============================================================================
# CELL 10 — Pre-computed ablation figures
# =============================================================================

import os
from IPython.display import Image, display, Markdown

ABLATION_DIR = "experiments/rl_finetuning/outputs/craftax_classic_ablations/figures"

KEY_FIGURES = [
    (
        "group_comparison.png",
        "**Group comparison** — final score by ablation group, against the "
        "pretrained checkpoint at **11.81**. Every group mean sits below it: "
        "**A** 9.74, **B** 7.07, **C** 10.53, **D** 7.97. Group C "
        "(architectural freezing) loses least, which is the plasticity "
        "ordering rather than anything about the reward: the conditions that "
        "keep the most of the checkpoint are the ones that change the fewest "
        "parameters.",
    ),
    (
        "score_delta_over_baseline_rl.png",
        "**Δ-score over baseline RL** — every condition, sorted. 17 of 25 "
        "finish above baseline RL (8.22) but **none of the 25 reaches the "
        "pretrained checkpoint**; the best, `lora`, stops 0.18 short at 11.63. "
        "The top of the range is occupied by the conditions that most restrict "
        "the update — `lora` 11.63, `trust_region_kl` 11.53, `head_only` "
        "11.30, `frozen_backbone` 11.09, `layer_ablation_top1` 10.93 — in "
        "roughly the order of how tightly they restrict it.",
    ),
    (
        "gradient_alignment.png",
        "**Gradient alignment** — cosine similarity between the "
        "return-weighted RL loss and the BC loss gradients over fine-tuning. "
        "The two gradients are taken at **different parameter points**, so "
        "this is a *retention diagnostic only* and not a measurement of the "
        "weighting: it confounds weighting with drift. It falls to −0.18 for "
        "baseline RL as the model leaves the checkpoint's basin, while `lora` "
        "— which barely moves — holds at 0.85. The separate "
        "single-parameter-point measurement is "
        "`‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine `0.02 ± 0.05`.",
    ),
    (
        "gradient_conflict_map.png",
        "**Per-layer gradient conflict** — where in the network the BC and "
        "return-weighted RL losses pull in opposite directions. Concentrated "
        "in the deeper backbone layers, which are the layers the Group C "
        "conditions freeze. That is why restricting the update caps how far "
        "the model can move, and why those conditions retain the most.",
    ),
]

for fname, caption in KEY_FIGURES:
    path = os.path.join(ABLATION_DIR, fname)
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(filename=path))
    else:
        display(Markdown(f"_Missing figure: `{fname}`_"))

In [ ]:
# =============================================================================
# CELL 11 — Ablation results tables (paper Tables 5 and 7)
# =============================================================================

import os
import polars as pl

TABLES_DIR = "experiments/rl_finetuning/outputs/craftax_classic_ablations/tables"

main_df = pl.read_csv(os.path.join(TABLES_DIR, "main_results.csv")).sort(
    "Final_Score", descending=True,
)
print(
    "Main ablation results (sorted by Final_Score).\n"
    "Pretrained checkpoint = 11.81; baseline RL = 8.22 (-3.59);\n"
    "best = lora at 11.63, still 0.18 below the checkpoint;\n"
    "worst = normalized_adv at 3.73.\n"
    "\n"
    "NOTE on the `Verdict` column: it is scored against *baseline RL*, not\n"
    "against the pretrained checkpoint. A row marked IMPROVEMENT beat the\n"
    "plain return-weighted objective; it did not beat the checkpoint it\n"
    "started from. Read Delta_vs_Pretrained for that — it is negative for\n"
    "all 25 conditions."
)
print(main_df)

group_df = pl.read_csv(os.path.join(TABLES_DIR, "group_summary.csv"))
print()
print("Group summary (every group mean is below the pretrained 11.81):")
print(group_df)

verdict_df = pl.read_csv(os.path.join(TABLES_DIR, "hypothesis_verdict.csv"))
print()
print("Hypothesis verdicts:")
print(verdict_df.select(["Ablation", "Group", "Result", "Conclusion"]))

with open(os.path.join(TABLES_DIR, "significance_test.txt")) as fh:
    print()
    print("Significance test (as shipped):")
    print(fh.read())

## 3. Conclusions

1. **Return-weighted ELBO fine-tuning degrades the checkpoint.** From **11.81**,
   baseline RL falls to **8.22 ± 0.14** and none of the 25 conditions ends above
   the checkpoint. The best result in the whole study is `lora` finishing **0.18
   short of never having trained**. The loss is concentrated where competence
   lives: tier-2 achievement rates lose a third and tier-3 more than half, with
   `make_iron_sword` falling 23% → 8%.
2. **It is not reward hacking under an evaluation mismatch.** Collection and
   evaluation use different denoising budgets and replanning intervals, so the
   decline is in principle compatible with the objective succeeding at what it
   optimises. It is not what happens — collected return falls from **5.21 to
   3.73** over the same iterations in which eval score falls from **12.06 to
   8.49**. The objective is failing on its own terms too.
3. **The reward is not what does the damage.** The return term is real and large:
   `‖g_δ‖/‖∇L_BC‖ = 0.49 ± 0.01` at cosine `0.02 ± 0.05`, against `0.893 ± 0.010`
   for `∇L_BC` against itself — around sixty standard deviations from a random
   direction, so it is structure and not noise. But the central control does not
   go our way. Advantage clipping cuts that ratio fivefold to `0.097 ± 0.003` and
   scores **3.16 below** baseline RL, second worst in the suite. Across the three
   transforms where the decomposition is defined, the ordering of the return
   term's magnitude is the reverse of the ordering of final score.
4. **What is left is the data.** The suite is ordered by plasticity — conditions
   that restrict the update keep more of the checkpoint (Spearman **−0.79**
   between final KL and final score) — and the condition closest to unweighted
   training on self-generated rollouts sits near the bottom. Together these point
   at fine-tuning on the model's own rollouts as the damaging ingredient, with the
   return weighting a large but incidental passenger. The paper does not dress
   this up as more than it is: without the unweighted arm it is an inference from
   a near-substitute.

**Why the sign constraint is not the whole story.** Eq. 3 is a bound only for
`A_i ≥ 0`, so the objective can re-rank sampled behaviour but never push mass away
from it, which caps how much *improvement* re-ranking can deliver. That does not
explain *degradation*, and it cannot be the whole story, because the same
non-negativity holds of reward-weighted and advantage-weighted regression, which
work elsewhere. The difference we can point to is the data: those methods are
usually applied to a fixed dataset rather than to rollouts fed back through a
denoising objective at every step.

**A second, separate cost of the surrogate.** Not offered as evidence for the
above: the ELBO surrogate is **badly conditioned** across diffusion time on Craftax
Classic. The gradient norm in the top third of the `t` range is **2.5×** the bottom
third, and the low-`t` and high-`t` gradient directions have mean cosine similarity
**0.06**, so the objective is dominated by the coarse-structure regime while what it
learns there is close to orthogonal to what it learns in the regime that fixes the
emitted tokens. On MiniHack the ratio is 1.14 and the cosine 0.52. Repairing the
conditioning does not recover the checkpoint: `low_t` (8.12) and `t_curriculum`
(8.41) sit at the baseline of 8.22 rather than above it.

**What these results do not establish.** There is no unweighted-ELBO-on-all-rollouts
arm and no continued-DAgger arm, so the attribution to self-generated data rests on
advantage clipping as a near-substitute rather than on a direct measurement. The null
characterises the clipped return ratio, not the exponential advantage weight
`exp(A/β)`. The gradient measurement is at one parameter point, on Craftax Classic
only, at one collection setting. MiniHack is not a second confirmation — the effect
there is small (47.5% → 43.8%, seed std 6.1) and three conditions finish nominally
above the checkpoint; it is reported because the suite is identical and the ordering
matches. The planner retains little zero-shot transfer (4.7% mean win rate on
held-out layouts against 48.5% in distribution), so no structural generalisation
claim is made. And the gradient-alignment cosine logged by the training harness takes
its two gradients at *different* parameter points, so it confounds weighting with
drift and is a **retention diagnostic only**, unrelated to the single-point
measurement above.

**Two cheap diagnostics, worth taking first.** For anyone reaching for this
objective: `CV_A` comes free from an effective-sample-size counter and says whether
the returns rank anything at all. The ratio `‖g_δ‖/‖∇L_BC‖` costs two backward
passes and says how much of the update the reward is responsible for. Ours said
0.49, which looked like a mechanism until the clipping control said otherwise.

**Open problem.** Per-step formulations over the denoising chain (d1, DiffPO) admit
**signed** advantages, and so change the object being optimised rather than the
weighting inside a regression. The obstacle for planners like ours is that those
estimators assume monotone unmasking, which ReMDM's inference-time remasking
violates. What these results argue for is an estimator that tolerates remasking
*and* admits a repulsive direction; **categorical flow matching** (Campbell et al.,
2024) is one route to the first half, admitting tractable log-likelihoods without an
ELBO surrogate.

The full project documentation lives in the HF repo's `README.md`.